# 16 UMAP / t-SNE Feature Visualization — Baseline vs SDD

**Question:** Does SDD produce more class-separable representations?

We project bottleneck features onto 2D with UMAP (preferred) or t-SNE
and compare baseline vs full SDD side-by-side.

If SDD clusters are tighter and better separated by class, it directly
validates the self-distillation hypothesis visually.

In [ ]:
from src.experiments import (
    load_cfg, deep_update, build_loaders,
    build_trainer_from_checkpoint,
    run_umap_comparison, run_tsne_comparison,
)

CIFAR10_CLASSES = ['airplane','automobile','bird','cat','deer',
                   'dog','frog','horse','ship','truck']

cfg_baseline = load_cfg('configs/cifar10.yaml')
cfg_baseline = deep_update(cfg_baseline, {
    'run_name': 'cifar10_mse_only',
    'sdd': {'enabled': False, 'use_centering': False, 'use_sharpening': False,
            'use_gating': False, 'use_projection_head': False, 'lambda_distill': 0.0},
})
cfg_sdd = load_cfg('configs/cifar10.yaml')
cfg_sdd = deep_update(cfg_sdd, {'run_name': 'cifar10_full_sdd'})

trainer_baseline = build_trainer_from_checkpoint(cfg_baseline, 'outputs/checkpoints/cifar10_mse_only_last.pt')
trainer_sdd      = build_trainer_from_checkpoint(cfg_sdd, 'outputs/checkpoints/cifar10_full_sdd_last.pt')

_, test_loader = build_loaders(cfg_sdd)

In [ ]:
# UMAP comparison (requires: pip install umap-learn)
trainers = {'Baseline (MSE only)': trainer_baseline, 'Full SDD': trainer_sdd}

try:
    umap_path = run_umap_comparison(
        trainers, test_loader,
        feature_layer='bottleneck',
        max_samples=2000,
        save_path='outputs/figures/umap_comparison.png',
        class_names=CIFAR10_CLASSES,
    )
    print('UMAP saved to:', umap_path)
except ImportError:
    print('umap-learn not installed. Run: pip install umap-learn')
    print('Falling back to t-SNE...')
    tsne_path = run_tsne_comparison(
        trainers, test_loader,
        feature_layer='bottleneck',
        max_samples=2000,
        save_path='outputs/figures/tsne_comparison.png',
        class_names=CIFAR10_CLASSES,
    )
    print('t-SNE saved to:', tsne_path)

In [ ]:
# Also compare across feature layers for the SDD model
from src.experiments import run_umap_comparison

# Wrap the same trainer under 4 different layer names (reuses one trained model)
# Note: we monkeypatch feature_layer temporarily for each projection
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from src.evaluation.feature_extract import extract_features
import torch

layers = ['bottleneck', 'skip1', 'skip2', 'decoder1']
fig, axes = plt.subplots(1, len(layers), figsize=(6 * len(layers), 5))

try:
    import umap
    import matplotlib.cm as cm
    import numpy as np

    for ax, layer in zip(axes, layers):
        feats, labels = extract_features(trainer_sdd.model, test_loader,
                                         trainer_sdd.device, feature_layer=layer)
        idx = torch.randperm(feats.shape[0])[:2000]
        f_np = feats[idx].numpy()
        l_np = labels[idx].numpy()

        emb = umap.UMAP(n_components=2, random_state=42).fit_transform(f_np)
        colors = cm.get_cmap('tab10', 10)
        for cls in range(10):
            mask = l_np == cls
            ax.scatter(emb[mask, 0], emb[mask, 1], s=5, alpha=0.6,
                       color=colors(cls), label=CIFAR10_CLASSES[cls])
        ax.set_title(f'layer: {layer}', fontsize=11)
        ax.set_xticks([]); ax.set_yticks([])

    axes[-1].legend(markerscale=2, fontsize=7, bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.suptitle('UMAP by feature layer (Full SDD model)', fontsize=13)
    plt.tight_layout()
    plt.savefig('outputs/figures/umap_layer_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
except ImportError:
    print('Install umap-learn to run this cell.')

In [ ]:
# t-SNE version (always available via sklearn)
tsne_path = run_tsne_comparison(
    trainers, test_loader,
    feature_layer='bottleneck',
    max_samples=2000,
    save_path='outputs/figures/tsne_comparison.png',
    class_names=CIFAR10_CLASSES,
)
print('t-SNE saved to:', tsne_path)